# Outcome Correlations

Pairwise associations among the three satisfaction outcomes analyzed in the
main models: `lifenow` (life satisfaction, 1-10), `satjob` (job satisfaction,
1-4), and `satfin` (financial satisfaction, 1-3). All three are ordinal, so
we report Spearman and Kendall in addition to Pearson; the rank-based
measures are the natural ones for these scales while Pearson is included
for reference.

Sample is restricted to rows where all three outcomes are observed and
in-range. This matches the analysis sample used by notebooks 07-09 in spirit
(those additionally require demographic predictors to be present), so the
correlations here use a slightly larger sample.


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import spearmanr, kendalltau, pearsonr

In [2]:
df = pd.read_csv('../data/gss_2022.csv')
outcomes = ['lifenow', 'satjob', 'satfin']

df_complete = df[outcomes].dropna()
df_complete = df_complete[
    df_complete['lifenow'].between(1, 10)
    & df_complete['satjob'].between(1, 4)
    & df_complete['satfin'].between(1, 3)
].reset_index(drop=True)

print(f'Complete-case sample: n = {len(df_complete)}')
print(df_complete.describe().round(2))

Complete-case sample: n = 1769
       lifenow   satjob   satfin
count  1769.00  1769.00  1769.00
mean      7.89     3.27     1.91
std       1.55     0.77     0.72
min       1.00     1.00     1.00
25%       7.00     3.00     1.00
50%       8.00     3.00     2.00
75%       9.00     4.00     2.00
max      10.00     4.00     3.00


## Correlation Matrices

Three methods side by side. Off-diagonal cells show the correlation and (in
parentheses) the p-value. p-values are based on each method's exact or
asymptotic distribution under the null of independence.

In [3]:
methods = {
    'Pearson': pearsonr,
    'Spearman': spearmanr,
    'Kendall': kendalltau,
}

corr_results = {}
pval_results = {}
for name, func in methods.items():
    rmat = np.zeros((3, 3))
    pmat = np.zeros((3, 3))
    for i, a in enumerate(outcomes):
        for j, b in enumerate(outcomes):
            r = func(df_complete[a], df_complete[b])
            rmat[i, j] = r.statistic if hasattr(r, 'statistic') else r[0]
            pmat[i, j] = r.pvalue if hasattr(r, 'pvalue') else r[1]
    corr_results[name] = pd.DataFrame(rmat, index=outcomes, columns=outcomes)
    pval_results[name] = pd.DataFrame(pmat, index=outcomes, columns=outcomes)

for name in methods:
    print(f'\n{name} correlation:')
    print(corr_results[name].round(3))
    print(f'{name} p-values:')
    print(pval_results[name].applymap(lambda p: f'{p:.2e}'))


Pearson correlation:
         lifenow  satjob  satfin
lifenow    1.000   0.296   0.379
satjob     0.296   1.000   0.246
satfin     0.379   0.246   1.000
Pearson p-values:
          lifenow    satjob    satfin
lifenow  0.00e+00  3.14e-37  2.22e-61
satjob   3.14e-37  0.00e+00  8.63e-26
satfin   2.22e-61  8.63e-26  0.00e+00

Spearman correlation:
         lifenow  satjob  satfin
lifenow    1.000   0.297   0.403
satjob     0.297   1.000   0.241
satfin     0.403   0.241   1.000
Spearman p-values:
          lifenow    satjob    satfin
lifenow  0.00e+00  2.16e-37  7.10e-70
satjob   2.16e-37  0.00e+00  9.27e-25
satfin   7.10e-70  9.27e-25  0.00e+00

Kendall correlation:
         lifenow  satjob  satfin
lifenow    1.000   0.255   0.347
satjob     0.255   1.000   0.219
satfin     0.347   0.219   1.000
Kendall p-values:
          lifenow    satjob    satfin
lifenow  0.00e+00  2.22e-36  2.29e-66
satjob   2.22e-36  0.00e+00  2.54e-24
satfin   2.29e-66  2.54e-24  0.00e+00


/tmp/ipykernel_1281118/2516219316.py:24: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  print(pval_results[name].applymap(lambda p: f'{p:.2e}'))
/tmp/ipykernel_1281118/2516219316.py:24: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  print(pval_results[name].applymap(lambda p: f'{p:.2e}'))
/tmp/ipykernel_1281118/2516219316.py:24: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  print(pval_results[name].applymap(lambda p: f'{p:.2e}'))


In [4]:
fig = make_subplots(rows=1, cols=3, subplot_titles=list(methods.keys()),
                    horizontal_spacing=0.08)

for k, name in enumerate(methods.keys(), start=1):
    mat = corr_results[name]
    fig.add_trace(
        go.Heatmap(
            z=mat.values,
            x=outcomes, y=outcomes,
            text=mat.round(3).values,
            texttemplate='%{text}',
            textfont=dict(size=14),
            colorscale='RdBu_r',
            zmin=-1, zmax=1,
            showscale=(k == 3),
            colorbar=dict(title='r', x=1.02) if k == 3 else None,
        ),
        row=1, col=k,
    )
    fig.update_yaxes(autorange='reversed', row=1, col=k)

fig.update_layout(
    title=f'Outcome correlation matrices (n = {len(df_complete)})',
    height=400, width=1200,
    plot_bgcolor='white',
)
fig.show()

## Pair Plots

Marginal distributions on the diagonal; joint distributions as count
heatmaps off-diagonal. Scatter plots are uninformative for these low-K
ordinal scales (heavy ties), so we use binned counts instead.

In [5]:
n = len(outcomes)
fig = make_subplots(
    rows=n, cols=n,
    shared_xaxes=False, shared_yaxes=False,
    horizontal_spacing=0.06, vertical_spacing=0.08,
)

for i, vi in enumerate(outcomes):
    for j, vj in enumerate(outcomes):
        row, col = i + 1, j + 1
        if i == j:
            counts = df_complete[vi].value_counts().sort_index()
            fig.add_trace(
                go.Bar(x=counts.index, y=counts.values,
                       marker_color='steelblue', showlegend=False),
                row=row, col=col,
            )
        else:
            ct = pd.crosstab(df_complete[vi], df_complete[vj])
            fig.add_trace(
                go.Heatmap(
                    z=ct.values,
                    x=ct.columns.astype(str), y=ct.index.astype(str),
                    text=ct.values, texttemplate='%{text}',
                    textfont=dict(size=10),
                    colorscale='Blues',
                    showscale=False,
                ),
                row=row, col=col,
            )
            fig.update_yaxes(autorange='reversed', row=row, col=col)
        if col == 1:
            fig.update_yaxes(title_text=vi, row=row, col=col)
        if row == n:
            fig.update_xaxes(title_text=vj, row=row, col=col)

fig.update_layout(
    title=f'Pairwise distributions of outcomes (n = {len(df_complete)})',
    height=750, width=900,
    plot_bgcolor='white',
)
fig.show()